# Kaggle Playground S6E11: LightGBM 5-Fold Stratified Baseline
**Competition:** [Predicting Credit Card Fraud](https://www.kaggle.com/competitions/playground-series-s6e11)  
**Metric:** ROC AUC  
**Target:** `is_fraud` (binary, heavily imbalanced ~1.67%)  

### Mục tiêu của notebook này:
1. Tự động nhận diện môi trường (**Kaggle Notebook** hay **máy Local**).
2. Tải và kiểm tra dữ liệu `train.csv`, `test.csv`.
3. Chia 5-Fold Stratified CV (seed=42) để đánh giá khách quan.
4. Huấn luyện mô hình **LightGBM** hỗ trợ biến Categorical tự nhiên.
5. Đánh giá điểm Out-Of-Fold (OOF) ROC-AUC.
6. Trực quan hóa Feature Importance.
7. Xuất đủ 3 file chuẩn quy ước nhóm: `oof.npy`, `test.npy`, `submission.csv`.


In [ ]:
import os
import gc
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f"LightGBM version: {lgb.__version__}")
print(f"Pandas version:   {pd.__version__}")


In [ ]:
# Cấu hình Thí nghiệm
EXP_ID = "lgbm_a_v1"
MEMBER = "A"
SEED = 42
N_SPLITS = 5

# Tự động nhận diện môi trường Kaggle vs Local
KAGGLE_INPUT = Path("/kaggle/input/playground-series-s6e11")
LOCAL_INPUT = Path("data/raw")

if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
    OUTPUT_DIR = Path("/kaggle/working")
    print("==> Đang chạy trên KAGGLE NOTEBOOK")
elif LOCAL_INPUT.exists():
    DATA_DIR = LOCAL_INPUT
    OUTPUT_DIR = Path("outputs")
    print("==> Đang chạy trên MÁY LOCAL")
else:
    raise FileNotFoundError("Không tìm thấy thư mục dữ liệu ở cả Kaggle và Local!")

# Tạo các thư mục lưu trữ output
(OUTPUT_DIR / "oof").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "preds").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "submissions").mkdir(parents=True, exist_ok=True)
print(f"Data dir:   {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
print("==> Đang đọc dữ liệu...")
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sub_df = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Sub shape:   {sub_df.shape}")

fraud_count = train_df['is_fraud'].sum()
fraud_rate = train_df['is_fraud'].mean()
print(f"Số mẫu gian lận: {int(fraud_count):,} / {len(train_df):,} ({fraud_rate:.4%})")


In [ ]:
ID_COL = "id"
TARGET = "is_fraud"

# Danh sách 5 biến categorical
CAT_FEATURES = [
    "merchant_category",
    "card_type",
    "auth_method",
    "channel",
    "device_type"
]

# Danh sách 6 biến boolean flags
BOOL_FEATURES = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt"
]

# Danh sách biến số
NUM_FEATURES = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes"
]

ALL_FEATURES = NUM_FEATURES + BOOL_FEATURES + CAT_FEATURES
print(f"Tổng số đặc trưng: {len(ALL_FEATURES)}")

# Chuẩn bị kiểu dữ liệu cho LightGBM
for col in CAT_FEATURES:
    train_df[col] = train_df[col].astype("category")
    test_df[col] = test_df[col].astype("category")

for col in BOOL_FEATURES:
    train_df[col] = train_df[col].astype(int)
    test_df[col] = test_df[col].astype(int)

X = train_df[ALL_FEATURES]
y = train_df[TARGET]
X_test = test_df[ALL_FEATURES]


In [ ]:
# Kiểm tra file folds.csv dùng chung của nhóm
FOLDS_FILE_LOCAL = Path("data/processed/folds.csv")
FOLDS_FILE_KAGGLE = Path("/kaggle/input/s6e11-team-artifacts/folds.csv")

if FOLDS_FILE_LOCAL.exists():
    print(f"==> Đọc fold từ {FOLDS_FILE_LOCAL}")
    folds_df = pd.read_csv(FOLDS_FILE_LOCAL)
    train_df["fold"] = folds_df["fold"]
elif FOLDS_FILE_KAGGLE.exists():
    print(f"==> Đọc fold từ {FOLDS_FILE_KAGGLE}")
    folds_df = pd.read_csv(FOLDS_FILE_KAGGLE)
    train_df["fold"] = folds_df["fold"]
else:
    print(f"==> Tự động tạo StratifiedKFold ({N_SPLITS} folds, seed={SEED})...")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    train_df["fold"] = -1
    for fold, (_, val_idx) in enumerate(skf.split(train_df, y)):
        train_df.loc[val_idx, "fold"] = fold

print("Phân bố số lượng và tỷ lệ gian lận theo fold:")
print(train_df.groupby("fold")[TARGET].agg(["count", "sum", "mean"]))


In [ ]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "n_estimators": 2500,
    "random_state": SEED,
    "n_jobs": -1,
    "verbose": -1
}


In [ ]:
oof_preds = np.zeros(len(train_df), dtype=np.float64)
test_preds = np.zeros(len(test_df), dtype=np.float64)
feature_importances = pd.DataFrame({"feature": ALL_FEATURES})
fold_aucs = []

start_time = time.time()
print(f"==> Bắt đầu huấn luyện {N_SPLITS}-Fold Cross Validation với LightGBM...
")

for fold in range(N_SPLITS):
    fold_start = time.time()
    
    train_idx = (train_df["fold"] != fold)
    val_idx = (train_df["fold"] == fold)
    
    X_train_f, y_train_f = X.loc[train_idx], y.loc[train_idx]
    X_val_f, y_val_f = X.loc[val_idx], y.loc[val_idx]
    
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    
    model.fit(
        X_train_f, y_train_f,
        eval_set=[(X_val_f, y_val_f)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )
    
    val_pred = model.predict_proba(X_val_f)[:, 1]
    oof_preds[val_idx] = val_pred
    
    fold_auc = roc_auc_score(y_val_f, val_pred)
    fold_aucs.append(fold_auc)
    
    # Cộng dồn test prediction
    test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS
    
    # Ghi nhận Feature Importance
    feature_importances[f"fold_{fold}"] = model.feature_importances_
    
    elapsed = time.time() - fold_start
    print(f"--- Fold {fold+1}/{N_SPLITS} | AUC: {fold_auc:.5f} | Best iteration: {model.best_iteration_} ({elapsed:.1f}s) ---
")

total_time = time.time() - start_time
overall_auc = roc_auc_score(y, oof_preds)

print("=" * 60)
print(f"Tổng thời gian: {total_time/60:.2f} phút")
print(f"AUC từng fold:  {[round(score, 5) for score in fold_aucs]}")
print(f"Trung bình AUC: {np.mean(fold_aucs):.5f} +/- {np.std(fold_aucs):.5f}")
print(f">> OVERALL OOF AUC: {overall_auc:.5f} <<")
print("=" * 60)


In [ ]:
# Tính trung bình feature importance
fold_cols = [f"fold_{f}" for f in range(N_SPLITS)]
feature_importances["importance"] = feature_importances[fold_cols].mean(axis=1)
feature_importances = feature_importances.sort_values(by="importance", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Feature Importance
sns.barplot(data=feature_importances.head(15), x="importance", y="feature", palette="Blues_r", ax=axes[0])
axes[0].set_title("Top 15 Most Important Features (LightGBM)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Average Split Importance")

# Subplot 2: ROC Curve
fpr, tpr, _ = roc_curve(y, oof_preds)
axes[1].plot(fpr, tpr, label=f"OOF ROC Curve (AUC = {overall_auc:.5f})", color="darkorange", lw=2)
axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Out-Of-Fold ROC Curve", fontsize=13, fontweight='bold')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


In [ ]:
# 1. Lưu OOF Predictions
oof_file = OUTPUT_DIR / "oof" / f"{EXP_ID}_oof.npy"
np.save(oof_file, oof_preds)
print(f"Đã lưu OOF: {oof_file} (shape={oof_preds.shape})")

# 2. Lưu Test Predictions
pred_file = OUTPUT_DIR / "preds" / f"{EXP_ID}_test.npy"
np.save(pred_file, test_preds)
print(f"Đã lưu Test Preds: {pred_file} (shape={test_preds.shape})")

# 3. Xuất file submission
sub_file = OUTPUT_DIR / "submissions" / f"{EXP_ID}.csv"
sub_df["is_fraud"] = test_preds
sub_df.to_csv(sub_file, index=False)
print(f"Đã lưu file nộp: {sub_file}")

# 4. Kiểm tra hợp lệ file submission
print("
--- Kiểm tra tính hợp lệ của file nộp ---")
print(f"Số dòng: {len(sub_df):,} (Kỳ vọng: 299,984)")
print(f"Số giá trị NaN: {sub_df['is_fraud'].isna().sum()}")
print(f"Khoảng giá trị xác suất: [{sub_df['is_fraud'].min():.5f}, {sub_df['is_fraud'].max():.5f}]")
print(f"Xác suất trung bình dự đoán: {sub_df['is_fraud'].mean():.4%}")
print("
5 dòng đầu:")
print(sub_df.head())


### Dòng kết quả ghi vào `experiments/results.csv`:
```csv
2026-11-12,lgbm_a_v1,A,LightGBM,raw_24f,5,[CV_AUC],[CV_STD],[PUBLIC_LB],notebooks/kaggle/02_baseline_lgbm.ipynb,baseline 5-fold stratified
```
